# cAPTure development OOF early-warning audit

This notebook reuses the completed operational OOF report and its selected score thresholds. It does not train models, read packet Parquet files, select new thresholds, or access final-test scenarios.

An attack-step iteration is timely only if the first correct window-close alert occurs strictly before its last malicious packet. For each development scenario, the first correct chain alert is early only if it occurs strictly before the first packet of a declared terminal action. Terminal actions are frozen in `configs/capture_early_warning_audit_v1.yaml`. Each scenario is one observed chain episode, so chain-level rates are descriptive.


## 1. Prepare Colab


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = [
    "code/python/requirements-capture-xgb.txt",
    "code/python/utils/capture_early_warning.py",
    "code/python/utils/capture_oof_operational.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_early_warning_audit_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


Mounted at /content/drive
Repository commit: cafe458d3dd3ea753a526a1abb6cf6e32f44bd01


## 2. Select the completed operational run

The default ID is the operational run recorded in the previous notebook. Change it if your completed run has another ID. Set `EARLY_WARNING_RUN_ID` only when resuming an existing audit.


In [2]:
from utils.capture_early_warning import (
    run_early_warning_audit, validate_early_warning_audit,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
POLICY_PATH = PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"
OPERATIONAL_RUN_ID = "20260920T142514_611048Z_operational_oof"
EARLY_WARNING_RUN_ID = None  # Set only when resuming an existing audit.
if EARLY_WARNING_RUN_ID is None:
    EARLY_WARNING_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_early_warning_oof"
    )
OPERATIONAL_DIR = DRIVE_ROOT / "operational_oof_runs" / OPERATIONAL_RUN_ID
OUTPUT_DIR = DRIVE_ROOT / "early_warning_oof_runs" / EARLY_WARNING_RUN_ID
print("Operational input:", OPERATIONAL_DIR)
print("Audit output:", OUTPUT_DIR)


Operational input: /content/drive/MyDrive/capture_gate0/operational_oof_runs/20260920T142514_611048Z_operational_oof
Audit output: /content/drive/MyDrive/capture_gate0/early_warning_oof_runs/20260920T160243_389215Z_early_warning_oof


## 3. Compute or verify the immutable audit


In [3]:
audit_args = {
    "operational_dir": OPERATIONAL_DIR,
    "manifest_path": MANIFEST_PATH,
    "policy_path": POLICY_PATH,
    "output_dir": OUTPUT_DIR,
}
if OUTPUT_DIR.exists():
    report = validate_early_warning_audit(**audit_args)
else:
    report = run_early_warning_audit(**audit_args)
operational = json.loads(
    (OPERATIONAL_DIR / "operational_report.json").read_text(encoding="utf-8")
)
print("Early-warning audit run ID:", EARLY_WARNING_RUN_ID)
print("Terminal actions:", report["policy"]["terminal_action_steps"])


Early-warning audit run ID: 20260920T160243_389215Z_early_warning_oof
Terminal actions: {'train_dollar_char': ['dollar_char'], 'train_slash_char': ['slash_char'], 'train_sub_exf': ['scp_exf'], 'train_empty_conn': ['empty_conn', 'empty_conn_ddos'], 'train_qos_mid': ['qos_mid', 'qos_mid_ddos']}


## 4. Compare warning times at every declared alert budget

The score-positive rate reproduces the original iteration detection rule. The timely rate requires the alert to arrive before the iteration's last malicious packet. The preterminal chain metric is a hierarchical macro of one episode-level flag per scenario; inspect the scenario table below rather than treating five episodes as a large sample.


In [4]:
comparison_rows = []
for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        audit_budget = model_report["budgets"][budget_name]
        operational_budget = operational["models"][model_name]["budgets"][budget_name]
        comparison_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": model_report["thresholds"][budget_name]["threshold"],
            "false_alert_windows_per_hour": (
                operational_budget["hierarchical_macro"]["false_alert_windows_per_hour"]),
            **audit_budget["hierarchical_macro"],
        })
comparison = pd.DataFrame(comparison_rows).set_index(["model", "budget"])
display(comparison)


threshold  false_alert_windows_per_hour  \
model          budget                                                       
xgb_p          one_per_hour        0.999463                      0.208772   
               one_per_12_hours    0.999942                      0.000000   
               one_per_5_minutes   0.987051                      3.163539   
current_window one_per_hour        0.983387                      0.658179   
               one_per_12_hours    0.994354                      0.000000   
               one_per_5_minutes   0.959686                      5.655613   
history        one_per_hour        0.995854                      0.918791   
               one_per_12_hours    0.999206                      0.036074   
               one_per_5_minutes   0.988180                      6.718528   
full           one_per_hour        0.997506                      0.433099   
               one_per_12_hours    0.998009                      0.036074   
               one_per_5_minutes   0.996420                      4.158372   

                                  score_positive_iteration_rate  \
model          budget                                             
xgb_p          one_per_hour                            0.528501   
               one_per_12_hours                        0.395452   
               one_per_5_minutes                       0.994527   
current_window one_per_hour                            0.909566   
               one_per_12_hours                        0.838935   
               one_per_5_minutes                       0.974778   
history        one_per_hour                            0.985104   
               one_per_12_hours                        0.860758   
               one_per_5_minutes                       0.995522   
full           one_per_hour                            0.844987   
               one_per_12_hours                        0.827461   
               one_per_5_minutes                       0.863420   

                                  timely_iteration_rate  \
model          budget                                     
xgb_p          one_per_hour                    0.395503   
               one_per_12_hours                0.281386   
               one_per_5_minutes               0.891673   
current_window one_per_hour                    0.847208   
               one_per_12_hours                0.824118   
               one_per_5_minutes               0.878904   
history        one_per_hour                    0.884298   
               one_per_12_hours                0.810621   
               one_per_5_minutes               0.894189   
full           one_per_hour                    0.827217   
               one_per_12_hours                0.817031   
               one_per_5_minutes               0.847334   

                                  late_positive_iteration_rate  \
model          budget                                            
xgb_p          one_per_hour                           0.132998   
               one_per_12_hours                       0.114067   
               one_per_5_minutes                      0.102855   
current_window one_per_hour                           0.062358   
               one_per_12_hours                       0.014817   
               one_per_5_minutes                      0.095874   
history        one_per_hour                           0.100806   
               one_per_12_hours                       0.050137   
               one_per_5_minutes                      0.101333   
full           one_per_hour                           0.017770   
               one_per_12_hours                       0.010430   
               one_per_5_minutes                      0.016087   

                                  early_before_terminal_action  
model          budget                                           
xgb_p          one_per_hour                                1.0  
               one_per_12_hours                            1.0  
    

## 5. Inspect individual scenarios and terminal-action deadlines


In [5]:
PRIMARY_BUDGET = "one_per_hour"
scenario_rows = []
chain_rows = []
for model_name, model_report in report["models"].items():
    audit_budget = model_report["budgets"][PRIMARY_BUDGET]
    for scenario, item in audit_budget["scenario_metrics"].items():
        scenario_rows.append({"model": model_name, **item})
    for scenario, item in audit_budget["chain_metrics"].items():
        chain_rows.append({"model": model_name, **item})
scenario_columns = [
    "fold", "iterations", "score_positive_iterations",
    "timely_iterations", "late_positive_iterations",
    "timely_iteration_rate", "early_before_terminal_action",
    "preterminal_opportunity_seconds",
]
display(pd.DataFrame(scenario_rows).set_index(["model", "scenario"])[scenario_columns])
chain_columns = [
    "first_terminal_action_step", "preterminal_opportunity_seconds",
    "first_correct_alert_steps", "early_before_terminal_action",
    "seconds_before_terminal_action",
    "seconds_at_or_after_terminal_action",
]
display(pd.DataFrame(chain_rows).set_index(["model", "scenario"])[chain_columns])


fold  iterations  score_positive_iterations  \
model          scenario                                                        
xgb_p          train_dollar_char    A         232                        172   
               train_slash_char     A         356                        295   
               train_sub_exf        A         335                        191   
               train_empty_conn     B         107                         46   
               train_qos_mid        B         171                         44   
current_window train_dollar_char    A         232                        226   
               train_slash_char     A         356                        355   
               train_sub_exf        A         335                        238   
               train_empty_conn     B         107                         96   
               train_qos_mid        B         171                        163   
history        train_dollar_char    A         232                        232   
               train_slash_char     A         356                        355   
               train_sub_exf        A         335                        306   
               train_empty_conn     B         107                        107   
               train_qos_mid        B         171                        171   
full           train_dollar_char    A         232                        220   
               train_slash_char     A         356                        352   
               train_sub_exf        A         335                        206   
               train_empty_conn     B         107                         82   
               train_qos_mid        B         171                        156   

                                  timely_iterations  late_positive_iterations  \
model          scenario                                                         
xgb_p          train_dollar_char                126                        46   
               train_slash_char                 251                        44   
               train_sub_exf                    102                        89   
               train_empty_conn                  41                         5   
               train_qos_mid                     28                        16   
current_window train_dollar_char                221                         5   
               train_slash_char                 348                         7   
               train_sub_exf                    194                        44   
               train_empty_conn                  86                        10   
               train_qos_mid                    156                         7   
history        train_dollar_char                225                         7   
               train_slash_char                 348                         7   
               train_sub_exf                    197                       109   
               train_empty_conn                  95                        12   
               train_qos_mid                    164                         7   
full           train_dollar_char                219                         1   
               train_slash_char                 348                         4   
               train_sub_exf                    199                         7   
               train_empty_conn                  82                         0   
               train_qos_mid                    148                         8   

                                  timely_iteration_rate  \
model          scenario                                   
xgb_p          train_dollar_char               0.543103   
               train_slash_char                0.705056   
               train_sub_exf                   0.304478   
               train_empty_conn                0.383178   
               train_qos_mid                   0.163743   
current_window train_dollar_char               0.952586   
               tr

first_terminal_action_step  \
model          scenario                                       
xgb_p          train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   
current_window train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   
history        train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   
full           train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   

                                  preterminal_opportunity_seconds  \
model          scenario                                             
xgb_p          train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   
current_window train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   
history        train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   
full           train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   

                                 first_correct_alert_steps  \
model          scenario                                      
xgb_p          train_dollar_char              [nmap_10_T5]   
               train_slash_char               [nmap_10_T5]   
               train_sub_exf                  [nmap_10_T5]   
               train_empty_conn               [nmap_10_T4]   
               train_qos_mid                 [nmap_banner]   
current_window train_dollar_char              [nmap_10_T5]   
               train_slash_char               [nmap_10_T5]   
               train_sub_exf                  [nmap_10_T5]   
               train_empty_conn               [nmap_10_T4]   
               train_qos_mid                 [nmap_banner]   
history        train_dollar_char              [nmap_10_T5]   
               train_slash_char               [nmap_10_T5]   
               train_sub_exf                  [nmap_10_T5]   
               train_empty_conn               [nmap_10_T4]   
               train_qos_mid                 [nmap_banner]   
full           train_dollar_char              [nmap_10_T5]   
           

## 6. Inspect attack steps with late or missing alerts


In [6]:
step_rows = []
for model_name, model_report in report["models"].items():
    for item in model_report["budgets"][PRIMARY_BUDGET]["step_metrics"].values():
        step_rows.append({"model": model_name, **item})
step_columns = [
    "model", "scenario", "attack_step", "iterations",
    "timely_iterations", "late_positive_iterations",
    "no_score_positive_iterations", "timely_iteration_rate",
]
step_table = pd.DataFrame(step_rows)[step_columns]
display(step_table.sort_values(
    ["timely_iteration_rate", "iterations"], ascending=[True, False]
).head(40))
print("Complete step and iteration details are stored in:",
      OUTPUT_DIR / "early_warning_report.json")


,model,scenario,attack_step,iterations,timely_iterations,late_positive_iterations,no_score_positive_iterations,timely_iteration_rate
23,xgb_p,train_sub_exf,scp_exf,117,0,0,117,0.000000
65,current_window,train_sub_exf,scp_exf,117,0,41,76,0.000000
39,xgb_p,train_qos_mid,qos_mid,39,0,0,39,0.000000
25,xgb_p,train_empty_conn,empty_conn,37,0,0,37,0.000000
18,xgb_p,train_sub_exf,mqtt_cat,27,0,0,27,0.000000
27,xgb_p,train_empty_conn,mqtt_cat,12,0,0,12,0.000000
32,xgb_p,train_empty_conn,sftp_inst,12,0,0,12,0.000000
74,current_window,train_empty_conn,sftp_inst,12,0,8,4,0.000000
116,history,train_empty_conn,sftp_inst,12,0,12,0,0.000000
158,full,train_empty_conn,sftp_inst,12,0,0,12,0.000000


Complete step and iteration details are stored in: /content/drive/MyDrive/capture_gate0/early_warning_oof_runs/20260920T160243_389215Z_early_warning_oof/early_warning_report.json


## Interpretation

A qualifying score after the last malicious packet is a late alert for that iteration. An alert after an earlier step can still be an early warning for the chain if it precedes the first declared terminal action. The chain table describes five development scenario episodes and does not estimate a population-level early-warning rate. The selected thresholds and OOF scores come from the same development data; final-test performance remains unmeasured.
